https://nlp.stanford.edu/projects/glove/

Jeffrey Pennington, Richard Socher, and Christopher D. Manning. 2014. GloVe: Global Vectors for Word Representation. [pdf] [bib]
Riley Carlson, John Bauer, and Christopher D. Manning. 2025. A New Pair of GloVes. [pdf]


In [71]:
import numpy as np


Check the file

In [8]:
file_path = 'wiki_giga_2024_300_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05_combined.txt'

with open(file_path, 'r') as file:
    for i in range(10):
        line = file.readline()
        if not line: 
            break
        print(line.strip())

the -0.15277 -0.092999 -0.229052 -0.447638 0.363178 0.10623300000000001 -0.139482 -0.10426900000000001 -0.002833000000000002 0.05523600000000001 0.180098 0.013903000000000006 -0.042463 -0.243767 0.296038 -0.345811 0.167596 0.483567 -0.13568599999999997 -0.09374700000000001 -0.043946999999999986 -0.186611 0.142375 0.089778 -0.278924 0.159325 -0.022184000000000002 -0.5068429999999999 0.121368 0.165918 -0.044317999999999996 0.06265100000000001 -0.175811 -0.224469 -0.15004599999999998 -0.331288 0.135372 0.053877999999999995 0.011467000000000005 -0.14016 0.004844000000000001 0.21540299999999998 -0.670909 0.394471 0.172528 0.234407 -0.051198 -0.026762999999999995 -0.015463000000000005 0.223858 -0.386525 0.144329 -0.427336 0.03425 -0.056078 0.229129 -0.10422900000000002 0.05115600000000001 0.07119199999999999 0.239731 0.485099 -0.374836 -0.503457 0.031803 -0.259148 0.649145 0.293875 -0.152335 -0.261509 0.13907199999999997 0.5473060000000001 0.19133299999999998 0.06903400000000001 0.161345 0.0

After a while of testing, it appears that there are vectors longer and shorter than 300 and also some non-castable values like ".".

I exclude the rows where any of these appear and keep count of how many rows we are deleting.

After the loop we see that 3533 rows have been excluded, which is fine considering the total row count is around 1.2 M.

In [47]:
def load_embeddings(file_path):
    embeddings = {}
    expected_dims = 300
    corrupted_lines_counter = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f, 1):
            values = line.split()

            if len(values) != expected_dims + 1:
                corrupted_lines_counter += 1
                continue
            
            word = values[0]
            
            try:
                vector = np.array(values[1:], dtype='float32')
                embeddings[word] = vector
            except ValueError:
                corrupted_lines_counter += 1
                continue
    
    print(corrupted_lines_counter)
    return embeddings


dictionary = load_embeddings(file_path)

3533


Simple function that returns matching 300 dim vector from the dictionary.

In [48]:
def get_vector(string):
    l_string = string.lower()

    if l_string in dictionary:
        return dictionary[l_string]

    print("Word not found")
    return


Testing that the vector math is working visually.

In [49]:
king_vec = get_vector("king")
woman_vec = get_vector("woman")
man_vec = get_vector("man")

new_vec = woman_vec - man_vec + king_vec
print(new_vec[:5])

[-0.184459 -0.423222  0.091106 -1.051715 -0.626441]


In [50]:
def cosine_similarity(v1, v2):
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    if norm_v1 == 0 or norm_v2 == 0:
        return 0
    return np.dot(v1, v2) / (norm_v1 * norm_v2)

Function that takes in the pre-calculated 300 dim vector and finds n closest vectors from the dictionary, excluding the words that was used in the calculation.

In [ ]:
def find_similar_words_with_vector_300(calculated_vector, words_used_in_calc: list, top_n):
    similar_words = {}
    input_words = words_used_in_calc

    print("Searching..")

    for word, vector in dictionary.items():
        if word in input_words:
            continue
        
        sim = cosine_similarity(calculated_vector, vector)
        
        # Just the first 10 words added straight
        if len(similar_words) < top_n:
            similar_words[word] = sim
            continue

        min_key = min(similar_words, key=similar_words.get)

        # Add only if cos sim bigger than smallest in the dict
        if sim > similar_words[min_key]:
            similar_words.pop(min_key)
            similar_words[word] = sim

    
    return sorted(similar_words.items(), key=lambda x: x[1], reverse=True)

Next we find similar words to "king" - "man" + "woman".

We see that our top match is "queen" with cosini similarity of ~0.73. It makes sense, since words "man" and "woman" share the same concept of being a gender, just like "king" and "queen" do.

When we subtract "man" from "king" it essentially takes the gender aspect away from the word "king". Then we add it back, but we instead of "man" we add "woman", which makes the "king" become a "queen".

In [ ]:
similar_words = find_similar_words_with_vector_300(new_vec, ["king", "man", "woman"], 10)
print(similar_words)

Searching..
[('queen', 0.7275747), ('princess', 0.6061351), ('throne', 0.5993189), ('monarch', 0.5972873), ('mother', 0.5888328), ('daughter', 0.58306766), ('elizabeth', 0.57210344), ('wife', 0.56494206), ('kingdom', 0.55931175), ('her', 0.5538283)]


In [58]:
alien_vec = get_vector("alien")

new_vec = woman_vec + man_vec + alien_vec

In [61]:
print(find_similar_words_with_vector_300(new_vec, ["woman", "man", "alien"], 10))

Searching..
[('girl', 0.7276618), ('person', 0.72703004), ('another', 0.6881981), ('boy', 0.68630517), ('one', 0.6692853), ('young', 0.6650222), ('who', 0.6619632), ('she', 0.6558699), ('life', 0.6553978), ('having', 0.64529824)]


In [67]:
small_vec = get_vector("small")
big_vec = get_vector("big")
tiny_vec = get_vector("tiny")

new_vec = big_vec - tiny_vec + small_vec

In [68]:
print(find_similar_words_with_vector_300(new_vec, ["big", "small", "tiny"], 10))

Searching..
[('large', 0.7276036), ('well', 0.6864056), ('major', 0.6797754), ('huge', 0.6691777), ('such', 0.66188335), ('like', 0.6568098), ('so', 0.6511364), ('because', 0.6494376), ('one', 0.64922), ('going', 0.64409405)]


In [69]:
dog_vec = get_vector("dog")
puppy_vec = get_vector("puppy")
cat_vec = get_vector("cat")

new_vec = dog_vec - puppy_vec + cat_vec

In [70]:
print(find_similar_words_with_vector_300(new_vec, ["dog", "puppy", "cat"], 10))

Searching..
[('dogs', 0.62154216), ('cats', 0.59120697), ('horse', 0.5665941), ('animal', 0.553318), ('pet', 0.5385772), ('wolf', 0.5034983), ('rat', 0.50271), ('animals', 0.50256234), ('bird', 0.48980796), ('lion', 0.48965222)]
